# Academic Research Agent System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/academic-research-agent/blob/main/demo_colab.ipynb)

**Multi-Agent System for Autonomous Academic Research Retrieval**

## Overview

This notebook demonstrates a multi-agent system based on the **BDI (Belief-Desire-Intention)** architecture that autonomously:
1. **Searches** for academic papers on ArXiv
2. **Extracts** metadata and content
3. **Processes** and filters by quality
4. **Stores** results in structured format

## Architecture

- **SearchAgent**: Discovers papers using structured queries
- **ExtractionAgent**: Extracts content from discovered sources
- **ProcessingAgent**: Filters and scores content quality
- **StorageAgent**: Persists results to disk

## 🔧 Setup (Run this first)

Install dependencies and clone the project:

In [ ]:
# Install required packages
!pip install requests beautifulsoup4 lxml -q

print("✓ Dependencies installed")

✓ Dependencies installed


In [1]:
# Option 1: Clone from GitHub (if you've pushed your code)
# !git clone https://github.com/YOUR_USERNAME/academic-research-agent.git
# %cd academic-research-agent

# Option 2: Create project structure in Colab
import os
os.makedirs('src/core', exist_ok=True)
os.makedirs('src/agents', exist_ok=True)
os.makedirs('src/pipeline', exist_ok=True)
os.makedirs('src/utils', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("✓ Project structure created")

✓ Project structure created


## 📤 Create Agent Code

In [2]:
%%writefile src/core/agent.py
"""
Base Agent class implementing BDI (Beliefs, Desires, Intentions) model.
"""
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, field


@dataclass
class Beliefs:
    """Represents agent's knowledge about its environment."""
    known_sources: List[str] = field(default_factory=list)
    keywords: List[str] = field(default_factory=list)
    context: Dict[str, Any] = field(default_factory=dict)

    def update(self, key: str, value: Any) -> None:
        self.context[key] = value

    def get(self, key: str, default: Any = None) -> Any:
        return self.context.get(key, default)


@dataclass
class Desires:
    """Represents agent's goals."""
    primary_goal: str
    sub_goals: List[str] = field(default_factory=list)
    constraints: Dict[str, Any] = field(default_factory=dict)


@dataclass
class Intentions:
    """Represents agent's committed plans/actions."""
    current_action: Optional[str] = None
    action_queue: List[str] = field(default_factory=list)
    completed_actions: List[str] = field(default_factory=list)

    def commit_action(self, action: str) -> None:
        self.current_action = action

    def complete_action(self) -> None:
        if self.current_action:
            self.completed_actions.append(self.current_action)
            self.current_action = None


class Agent(ABC):
    """Abstract base class for all agents implementing BDI architecture."""

    def __init__(self, agent_id: str, beliefs: Beliefs, desires: Desires):
        self.agent_id = agent_id
        self.beliefs = beliefs
        self.desires = desires
        self.intentions = Intentions()
        self.active = True

    @abstractmethod
    def perceive(self, environment: Dict[str, Any]) -> None:
        pass

    @abstractmethod
    def deliberate(self) -> None:
        pass

    @abstractmethod
    def act(self) -> Any:
        pass

    def run_cycle(self, environment: Dict[str, Any]) -> Any:
        self.perceive(environment)
        self.deliberate()
        return self.act()

    def get_status(self) -> Dict[str, Any]:
        return {
            "agent_id": self.agent_id,
            "active": self.active,
            "current_action": self.intentions.current_action,
            "completed_actions": self.intentions.completed_actions,
            "beliefs_context": self.beliefs.context
        }

Writing src/core/agent.py


In [7]:
%%writefile src/agents/search_agent.py
from typing import Any, Dict, List
import sys
sys.path.append('/content')
from src.core.agent import Agent, Beliefs, Desires


class SearchAgent(Agent):
    def __init__(self, agent_id: str = "search_agent"):
        beliefs = Beliefs(
            known_sources=["arxiv.org"],
            keywords=[]
        )
        desires = Desires(
            primary_goal="find_relevant_research_content",
            sub_goals=["identify_sources", "execute_queries", "filter_results"]
        )
        super().__init__(agent_id, beliefs, desires)
        self.search_results: List[Dict[str, str]] = []

    def perceive(self, environment: Dict[str, Any]) -> None:
        if "query" in environment:
            keywords = environment["query"].lower().split()
            self.beliefs.keywords = keywords
            self.beliefs.update("query", environment["query"])
        self.beliefs.update("max_results", environment.get("max_results", 10))

    def deliberate(self) -> None:
        if not self.beliefs.keywords:
            self.intentions.commit_action("wait_for_query")
            return
        self.intentions.commit_action("execute_search")

    def act(self) -> List[Dict[str, str]]:
        if self.intentions.current_action == "wait_for_query":
            self.intentions.complete_action()
            return []

        if self.intentions.current_action == "execute_search":
            query = self.beliefs.get("query", "")
            max_results = self.beliefs.get("max_results", 10)

            print(f"[SearchAgent] Searching ArXiv for: '{query}'")
            self.search_results = self._search_arxiv(query, max_results)
            self.intentions.complete_action()
            return self.search_results
        return []

    def _search_arxiv(self, query: str, max_results: int) -> List[Dict[str, str]]:
        import requests
        import xml.etree.ElementTree as ET
        from urllib.parse import quote
        import time

        # Simple query format
        parsed_query = f"all:{query}" if not any(f in query for f in ['ti:', 'au:', 'cat:']) else query

        base_url = "http://export.arxiv.org/api/query"
        safe_chars = ':[]"'
        encoded_query = quote(parsed_query, safe=safe_chars)
        search_query = f"search_query={encoded_query}&start=0&max_results={max_results}"
        url = f"{base_url}?{search_query}"

        try:
            time.sleep(3)  # Be nice to ArXiv
            response = requests.get(url, timeout=10)
            response.raise_for_status()

            root = ET.fromstring(response.content)
            ns = {'atom': 'http://www.w3.org/2005/Atom'}

            results = []
            for entry in root.findall('atom:entry', ns):
                paper_id = entry.find('atom:id', ns).text
                title = entry.find('atom:title', ns).text.strip().replace('\n', ' ')
                authors = [author.find('atom:name', ns).text
                          for author in entry.findall('atom:author', ns)]
                abstract = entry.find('atom:summary', ns).text.strip().replace('\n', ' ')
                published = entry.find('atom:published', ns).text[:10]

                pdf_link = None
                for link in entry.findall('atom:link', ns):
                    if link.get('title') == 'pdf':
                        pdf_link = link.get('href')
                        break

                results.append({
                    'url': paper_id,
                    'pdf_url': pdf_link,
                    'title': title,
                    'authors': authors,
                    'abstract': abstract,
                    'published': published,
                    'source': 'arxiv.org'
                })
            return results
        except Exception as e:
            print(f"[SearchAgent] Error: {e}")
            return []

Writing src/agents/search_agent.py


In [6]:
# Create simplified pipeline
import sys
sys.path.append('/content')
from src.agents.search_agent import SearchAgent
import json
from datetime import datetime

def run_research_pipeline(query, max_results=5):
    """Simplified pipeline for Colab."""
    print("\n" + "="*60)
    print("ACADEMIC RESEARCH PIPELINE")
    print("="*60 + "\n")

    # Search
    agent = SearchAgent()
    results = agent.run_cycle({"query": query, "max_results": max_results})

    if not results:
        print("No results found")
        return None

    print(f"✓ Found {len(results)} papers\n")

    # Process (simple filtering)
    processed = []
    for r in results:
        if len(r.get('abstract', '')) > 100:
            processed.append({
                **r,
                'publication_date': r['published'],
                'quality_score': 0.9
            })

    print(f"✓ Quality filtered to {len(processed)} papers\n")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"output/results_{timestamp}.json"

    with open(filename, 'w') as f:
        json.dump({"metadata": {"total_items": len(processed)}, "results": processed}, f, indent=2)

    print(f"✓ Saved to {filename}\n")
    return processed

print("✓ Pipeline ready")

ModuleNotFoundError: No module named 'src.agents.search_agent'

## Example 1: Simple Search

In [ ]:
# Search for neural networks papers
results = run_research_pipeline("neural networks", max_results=3)

# Display results
if results:
    for i, paper in enumerate(results, 1):
        print(f"\n{i}. {paper['title']}")
        print(f"   Authors: {', '.join(paper['authors'][:2])}")
        print(f"   Published: {paper['publication_date']}")
        print(f"   URL: {paper['url']}")
        if paper.get('pdf_url'):
            print(f"   PDF: {paper['pdf_url']}")
        print(f"   Abstract: {paper['abstract'][:200]}...")


ACADEMIC RESEARCH PIPELINE

[SearchAgent] Searching ArXiv for: 'neural networks'
✓ Found 3 papers

✓ Quality filtered to 3 papers

✓ Saved to output/results_20260210_184648.json


1. The Deep Arbitrary Polynomial Chaos Neural Network or how Deep Artificial Neural Networks could benefit from Data-Driven Homogeneous Chaos Theory
   Authors: Sergey Oladyshkin, Timothy Praditia
   Published: 2023-06-26
   URL: http://arxiv.org/abs/2306.14753v1
   PDF: https://arxiv.org/pdf/2306.14753v1
   Abstract: Artificial Intelligence and Machine learning have been widely used in various fields of mathematical computing, physical modeling, computational science, communication science, and stochastic analysis...

2. Learning Active Subspaces and Discovering Important Features with Gaussian Radial Basis Functions Neural Networks
   Authors: Danny D'Agostino, Ilija Ilievski
   Published: 2023-07-11
   URL: http://arxiv.org/abs/2307.05639v2
   PDF: https://arxiv.org/pdf/2307.05639v2
   Abstract: Providing

## Example 2: Structured Query

Search with field-specific syntax:

In [ ]:
# Title-only search
results = run_research_pipeline('ti:"transformer"', max_results=3)

if results:
    print(f"\n{'='*60}")
    print("RESULTS: Papers with 'transformer' in title")
    print(f"{'='*60}\n")

    for i, paper in enumerate(results, 1):
        print(f"{i}. {paper['title']}")
        print(f"   {', '.join(paper['authors'][:3])}")
        print()


ACADEMIC RESEARCH PIPELINE

[SearchAgent] Searching ArXiv for: 'ti:"transformer"'
✓ Found 3 papers

✓ Quality filtered to 3 papers

✓ Saved to output/results_20260210_184747.json


RESULTS: Papers with 'transformer' in title

1. Trainable Transformer in Transformer
   Abhishek Panigrahi, Sadhika Malladi, Mengzhou Xia

2. Transformer in Transformer
   Kai Han, An Xiao, Enhua Wu

3. DA-Transformer: Distance-aware Transformer
   Chuhan Wu, Fangzhao Wu, Yongfeng Huang



## Example 3: Category Search

Search in specific ArXiv category:

In [ ]:
# Machine Learning papers only
results = run_research_pipeline('cat:cs.LG', max_results=3)

if results:
    print(f"\nMachine Learning (cs.LG) Papers:")
    for i, paper in enumerate(results, 1):
        print(f"\n{i}. {paper['title']}")
        print(f"   PDF: {paper.get('pdf_url', 'N/A')}")


ACADEMIC RESEARCH PIPELINE

[SearchAgent] Searching ArXiv for: 'cat:cs.LG'
✓ Found 3 papers

✓ Quality filtered to 3 papers

✓ Saved to output/results_20260210_184842.json


Machine Learning (cs.LG) Papers:

1. Design Rule Checking with a CNN Based Feature Extractor
   PDF: https://arxiv.org/pdf/2012.11510v1

2. Unsupervised in-distribution anomaly detection of new physics through conditional density estimation
   PDF: https://arxiv.org/pdf/2012.11638v1

3. Detecting Botnet Attacks in IoT Environments: An Optimized Machine Learning Approach
   PDF: https://arxiv.org/pdf/2012.11325v1


## Try Your Own Query

Customize and run:

In [ ]:
# Modify this
my_query = "deep learning"
my_max_results = 3

results = run_research_pipeline(my_query, my_max_results)

# Your custom analysis here
if results:
    print(f"\nFound {len(results)} papers about: {my_query}")

## Download Results

Download the JSON files to your computer:

In [ ]:
from google.colab import files
import os

# List available result files
result_files = [f for f in os.listdir('output') if f.endswith('.json')]
print(f"Available files: {result_files}")

# Download the latest
if result_files:
    latest = sorted(result_files)[-1]
    files.download(f'output/{latest}')
    print(f"\nDownloaded: {latest}")

## Summary

This Colab notebook demonstrates:

✅ **Intelligent agent behavior** with BDI architecture  
✅ **Real ArXiv data retrieval**  
✅ **Structured query support**  
✅ **Quality filtering**  
✅ **Runs entirely in the cloud** (no local setup needed)  

### Supported Query Syntax:
- `ti:"neural networks"` - Search in title
- `au:Hinton` - Search by author
- `cat:cs.AI` - Search by category
- `ti:GAN AND cat:cs.CV` - Combined queries

### ArXiv Categories:
- `cs.AI` - Artificial Intelligence
- `cs.LG` - Machine Learning
- `cs.CV` - Computer Vision
- `cs.CL` - Natural Language Processing